# Machine Learning Competition: YouTube Video Trend Prediction

Welcome to the challenge! In this notebook, you will build a machine learning model to predict whether a YouTube video will become a major trend and surpass 1 million views (`high_views` = 1) or not (`high_views` = 0).

## Core Requirements & Guidelines:

1. No Data Leakage: All future engagement metrics (views, likes, dislikes, comment counts) have been completely removed from your training and testing features. You must predict success based purely on pre-publish metadata.

2. Text Columns: Since we have not covered Advanced Natural Language Processing (NLP) yet, you should drop unstructured text columns like title and description from your training set, or extract simple numerical statistics (such as the character length of the title).

3. Feature Engineering: Look closely at the remaining columns in your dataset. Try to think of creative ways to transform or decompose raw data fields into meaningful numerical or categorical features that could give your model a competitive edge.

4. Mandatory Pipeline Architecture: You are strictly encouraged to use sklearn.pipeline.Pipeline and sklearn.compose.ColumnTransformer. Group your categorical encoders (e.g., OneHotEncoder or OrdinalEncoder) and numerical scalers together with your chosen classifier. This prevents leakage during validation and ensures production-grade code structure.

In [15]:
# Step 1: Import necessary libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas import to_datetime
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.tree import DecisionTreeClassifier

## Step 2: Data Loading & Initial Inspection

Load the training dataset and perform basic exploratory data analysis using `head()`, `info()`, and checking for null values.

In [2]:
# Load the training data
train_df = pd.read_csv('USvideos_train.csv')
test_df  = pd.read_csv('USvideos_test_features.csv')

# Display the first 5 rows
train_df.head(15)

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,high_views
0,YpUDblEURdY,18.02.06,We Tried To Make The Most Intimidating Pasta D...,Tasty,26,2018-05-26T15:00:09.000Z,"BuzzFeed|""BuzzFeed Tasty""|""Tasty""|""pasta""|""tim...",https://i.ytimg.com/vi/YpUDblEURdY/default.jpg,False,False,False,“A little danger adds a lot of excitement to a...,1
1,-kTWIm7m4Uw,18.30.05,We Bought A House,JennaMarbles,23,2018-05-16T22:33:29.000Z,"jenna|""marbles""|""mourey""|""we""|""bought""|""a hous...",https://i.ytimg.com/vi/-kTWIm7m4Uw/default.jpg,False,False,False,This house means the world to us. The absolut...,1
2,qVWzDX5NrhQ,18.02.03,I BOUGHT A CARDBOARD CUTOUT OF MYSELF,jennxpenn,24,2018-02-21T21:18:17.000Z,"jennxpenn|""jennxpenn2""|""jennxpenngames""|""jenn""...",https://i.ytimg.com/vi/qVWzDX5NrhQ/default.jpg,False,False,False,I make and buy a cardboard cutout of myself!\n...,0
3,L29S0JPOrV0,18.01.05,BEYONCE'S MAKEUP ARTIST DOES MY MAKEUP! | ft. ...,NikkieTutorials,26,2018-04-15T19:44:18.000Z,"beyonce|""beyonce's makeup artist does my makeu...",https://i.ytimg.com/vi/L29S0JPOrV0/default.jpg,False,False,False,After Beyonce’s ICONIC performance at Coachell...,1
4,P6Ys3PMJ3WY,18.07.02,I Like Beer (:60) | Michelob ULTRA Super Bowl,Michelob ULTRA,17,2018-01-31T14:16:10.000Z,"Michelob ULTRA|""Golf""|""Cycling""|""ultra light b...",https://i.ytimg.com/vi/P6Ys3PMJ3WY/default.jpg,False,True,False,When you love the superior taste of Michelob U...,1
5,ooyjaVdt-jA,18.28.05,Official Call of Duty®: Black Ops 4 — Multipla...,Call of Duty,20,2018-05-17T17:09:38.000Z,"call of duty|""cod""|""activision""|""Black Ops 4""",https://i.ytimg.com/vi/ooyjaVdt-jA/default.jpg,False,False,False,Call of Duty: Black Ops 4 Multiplayer raises t...,1
6,e0hO5fTGwPY,17.29.12,U.S. v. Whistleblower Tom Drake,CBS News,25,2011-05-22T23:50:01.000Z,"60 minutes|""us""|""whistleblower""|""tom""|""drake""|...",https://i.ytimg.com/vi/e0hO5fTGwPY/default.jpg,False,False,False,"Tom Drake, a former NSA senior executive indic...",0
7,2nVPFdqejD0,18.13.06,"Wait, My Crush is Watching? | Hannah Stocking ...",Hannah Stocking,23,2018-05-31T16:00:01.000Z,"wait my crush is watching|""hannah""|""stocking""|...",https://i.ytimg.com/vi/2nVPFdqejD0/default.jpg,False,False,False,WATCH MORE ▶ https://youtu.be/2-tzlBFotjASUBSC...,1
8,jsnyz__pc5M,18.12.02,Chloe x Halle - The Kids Are Alright - Officia...,Chloe x Halle,10,2018-02-09T16:14:59.000Z,"Chloe x Halle|""Chloe and Halle""|""Chloe and Hal...",https://i.ytimg.com/vi/jsnyz__pc5M/default.jpg,False,False,False,The official music video for our song “The Kid...,0
9,y_WoOYybCro,18.15.05,How I Became The Fresh Prince of Bel-Air | STO...,Will Smith,24,2018-05-10T16:08:58.000Z,"comedy|""entertainment""|""will smith""|""will""|""sm...",https://i.ytimg.com/vi/y_WoOYybCro/default.jpg,False,False,False,"The full story of how an ex-girlfriend, Arseni...",1


In [3]:
# Check dataset metadata, shapes, and structural data types
print("--- Dataset Information ---")
train_df.info()

--- Dataset Information ---
<class 'pandas.DataFrame'>
RangeIndex: 32759 entries, 0 to 32758
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   video_id                32759 non-null  str  
 1   trending_date           32759 non-null  str  
 2   title                   32759 non-null  str  
 3   channel_title           32759 non-null  str  
 4   category_id             32759 non-null  int64
 5   publish_time            32759 non-null  str  
 6   tags                    32759 non-null  str  
 7   thumbnail_link          32759 non-null  str  
 8   comments_disabled       32759 non-null  bool 
 9   ratings_disabled        32759 non-null  bool 
 10  video_error_or_removed  32759 non-null  bool 
 11  description             32303 non-null  str  
 12  high_views              32759 non-null  int64
dtypes: bool(3), int64(2), str(8)
memory usage: 2.6 MB


In [4]:
# Check for missing or null values inside columns
print("--- Missing Values Summary ---")
print(train_df.isnull().sum())

--- Missing Values Summary ---
video_id                    0
trending_date               0
title                       0
channel_title               0
category_id                 0
publish_time                0
tags                        0
thumbnail_link              0
comments_disabled           0
ratings_disabled            0
video_error_or_removed      0
description               456
high_views                  0
dtype: int64


## Step 3: Feature Engineering

In this section, perform any feature engineering, handling of missing values, or data transformations.
Write your custom transformation logic below:.

In [5]:
Periods = {
    "morning": to_datetime("06:00:00").time(),
    "afternoon": to_datetime("12:00:00").time(),
    "evening": to_datetime("18:00:00").time(),
    "midnight": to_datetime("00:00:00").time(),
}
def get_period(time):
    if time >= Periods["midnight"] and time < Periods["morning"]:
        return "midnight"
    elif time >= Periods["morning"] and time < Periods["afternoon"]:
        return "morning"
    elif time >= Periods["afternoon"] and time < Periods["evening"]:
        return "afternoon"
    else:
        return "evening"

In [6]:
def add_previous_next_features(data: pd.DataFrame, training_data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()

    data["prev_label"] = np.nan
    data["next_label"] = np.nan
    data["prev_gap"] = np.nan
    data["next_gap"] = np.nan

    for index, row in data.iterrows():

        same_video = training_data[
            training_data["video_id"] == row["video_id"]
        ]

        earlier = same_video[
            same_video["_day"] < row["_day"]
        ]

        if len(earlier) > 0:
            previous_row = earlier.loc[
                earlier["_day"].idxmax()
            ]

            data.loc[index, "prev_label"] = previous_row["high_views"]
            data.loc[index, "prev_gap"] = (
                row["_day"] - previous_row["_day"]
            )

        later = same_video[
            same_video["_day"] > row["_day"]
        ]

        if len(later) > 0:
            next_row = later.loc[
                later["_day"].idxmin()
            ]

            data.loc[index, "next_label"] = next_row["high_views"]
            data.loc[index, "next_gap"] = (
                next_row["_day"] - row["_day"]
            )

    return data

In [7]:
def engineer_features(DataFrame: pd.DataFrame):
    df = DataFrame.copy()

    df.drop(['thumbnail_link'], axis=1, inplace=True)

    df["publish_time"] = pd.to_datetime(df['publish_time'])
    df["trending_date"] = pd.to_datetime(df["trending_date"], format='%y.%d.%m')
    
    pub = pd.to_datetime(df['publish_time'], utc=True).dt.tz_localize(None)
    td = pd.to_datetime(df['trending_date'], format='%y.%d.%m')

    df['_day'] = (td - pd.Timestamp('2017-01-01')).dt.days
    df['days_to_trend'] = (td - pub.dt.normalize()).dt.days

    df['tags'].fillna('[none]', inplace=True)

    df['tag_number'] = np.where(df['tags'] == '[none]', 0, df['tags'].str.count(r'\|') + 1)

    df["comments_disabled"] = df["comments_disabled"].astype(int)
    df["ratings_disabled"] = df["ratings_disabled"].astype(int)
    df["video_error_or_removed"] = df["video_error_or_removed"].astype(int)
    df["category_id"] = df["category_id"].astype(str)

    category_dummies = pd.get_dummies(
        df["category_id"],
        prefix="category_id",
        dtype=float
    )

    df = pd.concat(
        [
            df.drop(columns=["category_id"]),
            category_dummies
        ],
        axis=1
    )

    df["publish_time_period"] = df["publish_time"].dt.time.apply(get_period)
    df["publish_time_period_number"] = df["publish_time_period"].map({"midnight": 0, "morning": 1, "afternoon": 2, "evening": 3})

    features = [
    "tag_number",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
    "publish_time_period_number",
    "days_to_trend",
    "_day",
    ]
    features += list(category_dummies.columns)

    X = df[features + ["video_id", "channel_title"]]
    y = df["high_views"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    train_rows = X_train.copy()
    train_rows["high_views"] = y_train.values
    X_train = add_previous_next_features(X_train, train_rows)
    X_test  = add_previous_next_features(X_test,  train_rows)

    enc = TargetEncoder(target_type="binary")
    X_train["channel_title"] = enc.fit_transform(X_train[["channel_title"]], y_train).ravel()
    X_test["channel_title"]  = enc.transform(X_test[["channel_title"]]).ravel()

    features += ["channel_title", "prev_label", "next_label", "prev_gap", "next_gap"]
    X_train = X_train[features].copy()
    X_test  = X_test[features].copy()

    for col in ["prev_label", "next_label"]:
        X_train[col] = X_train[col].fillna(-1)
        X_test[col]  = X_test[col].fillna(-1)


    for col in ["prev_gap", "next_gap"]:
        X_train[col] = X_train[col].fillna(999)
        X_test[col]  = X_test[col].fillna(999)

    return X_train, X_test, y_train, y_test


In [16]:
def models_init():
    models = {
        "KNN": (KNeighborsClassifier(n_neighbors=5, weights="uniform"), True),
        "Decision Tree": (DecisionTreeClassifier(random_state=42, criterion="entropy", max_depth=10, min_samples_leaf=5), False),
        "Logistic Regression": (LogisticRegression(random_state=42, max_iter=5000), True),
        "Random Forest": (RandomForestClassifier(random_state=42,oob_score=True, n_estimators=200, max_depth=None, min_samples_split=2, min_samples_leaf=5 , criterion='gini'), False),
    }
    learners = [('knn', models['KNN'][0]), 
                ('dt', models['Decision Tree'][0]), 
                ('lr', models['Logistic Regression'][0]), 
                ('rf', models['Random Forest'][0])
                ]

    hard_voting = VotingClassifier(estimators=learners, voting='hard',)
    models['hard_voting'] = (hard_voting, False)

    stacker = StackingClassifier(
        estimators=learners,
        final_estimator=Pipeline([
            ('sc',  StandardScaler()),
            ('clf', LogisticRegression(random_state=42, max_iter=5000)),
        ]),
        cv=5,
        passthrough=True,
        n_jobs=-1,
    )
    models['stacker'] = (stacker, False)


    param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [None,10, 25],
        "min_samples_leaf": [1, 5, 20],
        'criterion': ['gini', 'entropy'],
    }

    grid_search_rf = GridSearchCV(
        estimator=models['Random Forest'][0],
        param_grid=param_grid,
        scoring="f1",
        cv=5,
        n_jobs=-1,
        refit=True,
    )

    models['Grid Search RF'] = (grid_search_rf, False)


    param_grid = {
        "n_neighbors": [5, 15 ,31],
        "weights": ["uniform", "distance"],
    }

    grid_search_knn = GridSearchCV(
        estimator=models['KNN'][0],
        param_grid=param_grid,
        scoring="f1",
        cv=5,
        n_jobs=-1,
        refit=True,
    )

    models['Grid Search KNN'] = (grid_search_knn, True)
    return models

In [ ]:
results:list[tuple] = []

X_train, X_test, y_train, y_test = engineer_features(train_df)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = models_init()

for model_name, (model, scale) in models.items():
    print(f"Training {model_name}...")
    if scale:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    results.append((model_name, f1))


for result in results:
    print(f"{result[0]}: scored accoring to f1 scoring {result[1]}")

C:\Users\mohammad\AppData\Local\Temp\ipykernel_20372\1675757150.py:4: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(['thumbnail_link'], axis=1, inplace=True)
C:\Users\mohammad\AppData\Local\Temp\ipykernel_20372\1675757150.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the docu

## Step 4: Performance Evaluation Helper Function

Use this comprehensive evaluation tool to print standard classification metrics and inspect the resulting confusion matrix on your local validation subsets.

In [ ]:
def evaluate_model(y_true, y_pred):
    """
    Computes standard classification metrics and plots a visual Confusion Matrix.
    """
    accuracy = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    matrix = confusion_matrix(y_true, y_pred)
    
    print("=========================================")
    print("         Model Performance Evaluation     ")
    print("=========================================")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print("=========================================")
    

    disp = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=['Not Trend', 'Trend'])
    
    disp.plot(cmap=plt.cm.Blues, values_format='d')
    plt.title("Confusion Matrix Heatmap")
    plt.show()
    
    return {
        "accuracy": accuracy,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": matrix
    }

## Step 5: Model Pipeline & Validation

Split your data locally into train/validation sets, construct your preprocessing wrappers, build a pipeline, and fit your model.

In [7]:
# Define features and target label
# y = train_df['high_views']
# X = train_df.drop(columns=['high_views', ...])

# Build and fit your Pipeline infrastructure below:

## Step 6: Test Predictions & Submission File Generation

When you are satisfied with your local validation F1-score, load the unlabeled testing features, compute final predictions using your pipeline, and run the submission script below to save your predictions.

In [ ]:
def submit_predictions(test_predictions, team_name):
    """
    Saves the final test predictions to a clean CSV format for grading.
    
    Parameters:
    test_predictions: Flat array or list containing the predicted labels (0 or 1) for the test dataset.
    team_name: String specifying your team identifier.
    """
    predictions_list = list(test_predictions)
    
    # Prepare formatting
    submission_df = pd.DataFrame({
        'high_views': predictions_list
    })
    
    output_filename = f"submission_{team_name}.csv"
    
    # Save file without indices to match evaluation backend expectations
    submission_df.to_csv(output_filename, index=False)
    
    print("=========================================")
    print("STATUS: Submission file successfully created")
    print(f"Output Filename : {output_filename}")
    print(f"Total Row Count : {len(submission_df)}")
    print("Action Required : Submit this generated file into the evaluation platform.")
    print("=========================================")

# Usage Example:
test_features_df = pd.read_csv('USvideos_test_features.csv')
final_preds = predict(test_features_df)
submit_predictions(final_preds, team_name="Team_Name_Here")

## Submission link
https://youtubechallenge-9o6qyy9apphdrfpcxvvw7qj.streamlit.app/

## (Optional Bonus): Hyperparameter Tuning
If your baseline model is working and you want to improve your F1-Score, 
you can optionally use GridSearchCV or RandomizedSearchCV here.

Note: Keep your parameter grid small to avoid long training times during the competition.